<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/advanced_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone --recursive --branch zeynep-september https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git \
/content/code

Cloning into '/content/code'...
remote: Enumerating objects: 876, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 876 (delta 99), reused 92 (delta 88), pack-reused 750 (from 1)
Receiving objects: 100% (876/876), 96.49 MiB | 6.57 MiB/s, done.
Resolving deltas: 100% (473/473), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 3.11 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [4]:
%cd /content/code

/content/code


In [5]:
import sys
import torch
import torchvision

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
CUDA version: 12.8


In [6]:
!pip install -q torcheval pyrebase4 yacs loguru wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [4]:
import os
import subprocess

DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
FEATURES_ZIP = f"{DRIVE_BASE_PATH}/1s.zip"

CODE_DIR = "/content/code"
TEMP_DIR = "/content/temp_features"
FEATURE_DIR = f"{CODE_DIR}/data/features"

# clean temp area
!rm -rf "{TEMP_DIR}"
!mkdir -p "{TEMP_DIR}"
!mkdir -p "{FEATURE_DIR}"

# copy the outer zip locally
!cp "{FEATURES_ZIP}" /content/1s.zip

# unzip outer archive
!unzip -q -o /content/1s.zip -d "{TEMP_DIR}"

# IMPORTANT:
# omnivore.zip already contains an omnivore/ folder,
# so extract into data/features, NOT data/features/omnivore
!unzip -q -o "{TEMP_DIR}/1s/video/omnivore.zip" -d "{FEATURE_DIR}"

# clean temp files
!rm /content/1s.zip
!rm -rf "{TEMP_DIR}"

print("Omnivore features ready.")

Omnivore features ready.


In [5]:
files = os.listdir("/content/code/data/features/omnivore")
print("Number of Omnivore feature files:", len(files))
print(files[:5])


Number of Omnivore feature files: 384
['22_38_360p.mp4_1s_1s.npz', '25_109_360p.mp4_1s_1s.npz', '9_15_360p.mp4_1s_1s.npz', '28_7_360p.mp4_1s_1s.npz', '26_42_360p.mp4_1s_1s.npz']


In [7]:
DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"

CHECKPOINT_ZIP = f"{DRIVE_BASE_PATH}/error_recognition_best.zip"

In [8]:
!unzip -l "$CHECKPOINT_ZIP" | head -30

Archive:  /content/drive/MyDrive/AML_Project/error_recognition_best.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   825896  2024-05-20 17:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_recordings_epoch_45.pt
   825904  2024-05-20 19:41   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_environment_epoch_11.pt
   825784  2024-05-20 20:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_step_epoch_41.pt
   825800  2024-05-20 18:51   error_recognition_best/MLP/3dresnet/error_recognition_MLP_3dresnet_person_epoch_39.pt
  2103976  2024-05-21 19:16   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_environment_epoch_50.pt
  2103864  2024-05-21 17:51   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_person_epoch_8.pt
  2103856  2024-05-21 04:28   error_recognition_best/MLP/imagebind/error_recognition_MLP_imagebind_audio_step_epoch_28.pt
  2103960  20

In [9]:
import os

CHECKPOINT_DIR = "/content/code/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!cp "$CHECKPOINT_ZIP" /content/checkpoints.zip
!unzip -q -o /content/checkpoints.zip -d "$CHECKPOINT_DIR"
!rm /content/checkpoints.zip

print("Checkpoints extracted.")

Checkpoints extracted.


In [10]:
ckpt = "/content/code/checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt"

print(os.path.exists(ckpt))

True


In [11]:
import os

print("annotations:", os.path.exists("/content/code/annotations/annotation_json/step_annotations.json"))
print("features:", os.path.exists("/content/code/data/features/omnivore"))
print("checkpoint:", os.path.exists("/content/code/checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt"))

annotations: False
features: True
checkpoint: True


In [14]:
import glob
import os
import subprocess

configs = [
    {
        "variant": "MLP",
        "backbone": "omnivore",
        "split": "step",
        "threshold": 0.6,
        "pattern": "*MLP*omnivore*step*.pt"
    },
    {
        "variant": "MLP",
        "backbone": "omnivore",
        "split": "recordings",
        "threshold": 0.4,
        "pattern": "*MLP*omnivore*recordings*.pt"
    },
    {
        "variant": "Transformer",
        "backbone": "omnivore",
        "split": "step",
        "threshold": 0.6,
        "pattern": "*Transformer*omnivore*step*.pt"
    },
    {
        "variant": "Transformer",
        "backbone": "omnivore",
        "split": "recordings",
        "threshold": 0.4,
        "pattern": "*Transformer*omnivore*recordings*.pt"
    },
]

base_ckpt_dir = "/content/code/checkpoints/error_recognition_best"
code_dir = "/content/code"

print("Starting Omnivore baseline reproduction...")

for conf in configs:
    variant = conf["variant"]
    backbone = conf["backbone"]
    split = conf["split"]
    threshold = conf["threshold"]

    search_path = os.path.join(
        base_ckpt_dir,
        variant,
        backbone,
        conf["pattern"]
    )

    files = glob.glob(search_path)

    if not files:
        print(f"SKIPPING: No checkpoint for {variant} ({backbone}) on {split}")
        continue

    ckpt_path = os.path.relpath(files[0], code_dir)

    print("\n" + "=" * 60)
    print(f"{variant} ({backbone}) | {split} | threshold={threshold}")
    print(f"Checkpoint: {ckpt_path}")
    print("=" * 60)

    cmd = [
        "python", "-m", "core.evaluate",
        "--variant", variant,
        "--backbone", backbone,
        "--ckpt", ckpt_path,
        "--split", split,
        "--threshold", str(threshold),
    ]

    result = subprocess.run(
        cmd,
        cwd=code_dir,
        text=True
    )

    if result.returncode != 0:
        print(f"Evaluation failed for {variant} / {split}")

print("\nReproduction sequence complete.")

Starting Omnivore baseline reproduction...

MLP (omnivore) | step | threshold=0.6
Checkpoint: checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt
Evaluation failed for MLP / step

MLP (omnivore) | recordings | threshold=0.4
Checkpoint: checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_recordings_epoch_33.pt
Evaluation failed for MLP / recordings

Transformer (omnivore) | step | threshold=0.6
Checkpoint: checkpoints/error_recognition_best/Transformer/omnivore/error_recognition_Transformer_omnivore_step_epoch_9.pt
Evaluation failed for Transformer / step

Transformer (omnivore) | recordings | threshold=0.4
Checkpoint: checkpoints/error_recognition_best/Transformer/omnivore/error_recognition_Transformer_omnivore_recordings_epoch_31.pt
Evaluation failed for Transformer / recordings

Reproduction sequence complete.
